In [1]:
from pathlib import Path

import json

import hashlib

In [2]:
BASE = Path("/workspace/tiny-llm-from-scratch")

INPUT = BASE / "datasets/processed/claude_mythos/normalized_dataset.json"

OUTPUT = BASE / "datasets/processed/claude_mythos/deduplicated_dataset.json"

REPORT = BASE / "datasets/reports/duplicate_report.json"

REPORT.parent.mkdir(parents=True, exist_ok=True)

In [3]:
with open(INPUT, encoding="utf-8") as f:

    dataset = json.load(f)

print(len(dataset))

25000


In [4]:
def sample_hash(sample):

    text = (
        sample["user"]
        + "\n"
        + sample["assistant"]
    )

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

In [5]:
seen = set()

unique = []

duplicates = 0

for sample in dataset:

    h = sample_hash(sample)

    if h in seen:

        duplicates += 1

        continue

    seen.add(h)

    unique.append(sample)

print("Unique Samples :", len(unique))

print("Duplicates Removed :", duplicates)

Unique Samples : 5315
Duplicates Removed : 19685


In [6]:
with open(OUTPUT, "w", encoding="utf-8") as f:

    json.dump(

        unique,

        f,

        indent=2,

        ensure_ascii=False

    )

print(OUTPUT)

/workspace/tiny-llm-from-scratch/datasets/processed/claude_mythos/deduplicated_dataset.json


In [7]:
report = {

    "original_samples": len(dataset),

    "unique_samples": len(unique),

    "duplicates_removed": duplicates,

    "duplicate_percentage": round(
        duplicates / len(dataset) * 100,
        2
    )

}

with open(REPORT, "w", encoding="utf-8") as f:

    json.dump(

        report,

        f,

        indent=4

    )

print(REPORT)

/workspace/tiny-llm-from-scratch/datasets/reports/duplicate_report.json


In [8]:
with open(OUTPUT, encoding="utf-8") as f:

    verify = json.load(f)

print(len(verify))

print(verify[0].keys())

5315
dict_keys(['user', 'assistant', 'category'])


In [9]:
print("=" * 70)

print("DUPLICATE REMOVAL REPORT")

print("=" * 70)

for key, value in report.items():

    print(f"{key:25} : {value}")

print("=" * 70)

print("Duplicate Removal Completed Successfully")

print("=" * 70)

DUPLICATE REMOVAL REPORT
original_samples          : 25000
unique_samples            : 5315
duplicates_removed        : 19685
duplicate_percentage      : 78.74
Duplicate Removal Completed Successfully
